# ChipFF Bench: thermal expansion on a free GPU (Kaggle)

Runs the GPU part of ChipFF Bench: constant-pressure MD in kUPS for every material x model. The CPU parts
(lattice constants, elastic tensors) are already inside the project under `results/raw/`.

**One-time setup (Kaggle)**
1. Verify your phone number (Settings → Phone verification). Kaggle needs this before notebooks can use the internet.
2. Notebook: File → Import notebook → this file. It downloads the code from the public GitHub repo, so no dataset is needed
   (alternatively upload `chipff-bench.zip` from `scripts/make_kaggle_bundle.sh` as a dataset and set `GITHUB_REPO = ""`).
3. Right panel: Session options → Accelerator **GPU T4 x2**, Internet **On**. The notebook runs one model per GPU in parallel.
4. Optional, for UMA: Add-ons → Secrets → add `HF_TOKEN` (a Hugging Face read token whose account has UMA access).

**Run it unattended:** click **Save Version → Save & Run All (Commit)**. It runs in the background (you can close the tab),
up to ~11 h, and saves each finished batch. When it ends, open the version → **Output** → download `chipff_thermal_results.zip`.

**If it didn't finish everything (likely on the first run):** in the notebook's right panel, **Add Input → Your Work → Notebooks**,
pick this notebook (its latest version's output), then Save & Run All again. It loads the finished results and continues
with the rest. Repeat until the status table at the end shows every material.

## 1. Settings

In [ ]:
MODELS = ["mace-mpa-0", "orb-v3", "uma-s-1p2"]   # MatterSim is run without kUPS and has no thermal step
RUN_PILOT = False         # optional quick check (2 models x 4 materials); the full run logs timings per batch anyway
RUN_FULL = True
RUN_ROBUSTNESS = True     # after each model's main run: 2 extra seeds (6 materials), 14 A cells and half time step (3)
TIME_BUDGET_H = 11.0      # stop starting new models after this (Kaggle sessions end at 12 h)
GITHUB_REPO = "https://github.com/anushajain2/chipff-bench"   # set to "" to use an attached dataset instead

## 2. Get the code and install (Python 3.12 via uv, since kUPS needs Python ≥ 3.12)

In [ ]:
import glob, os, shutil, subprocess, time
T0 = time.time()
!nvidia-smi --query-gpu=name,memory.total --format=csv
OUT = "/kaggle/working" if os.path.exists("/kaggle") else os.path.abspath(".")
WORK = "/tmp/chipff-bench"   # not saved by Kaggle, so the version output stays small
if not os.path.exists(WORK):
    if GITHUB_REPO:
        subprocess.run(["git", "clone", "-q", GITHUB_REPO, WORK], check=True)
    else:
        hits = glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
        assert hits, "Attach the chipff-bench dataset (Add input) or set GITHUB_REPO"
        shutil.copytree(os.path.dirname(hits[0]), WORK)
%cd {WORK}
# resume: load thermal results from a previous run attached as an input (Add Input -> this notebook's output)
import zipfile
prev = glob.glob("/kaggle/input/**/chipff_thermal_results.zip", recursive=True)
for z in prev:
    with zipfile.ZipFile(z) as f:
        names = [n for n in f.namelist() if n.startswith("results/raw/") and n.endswith(".json")]
        f.extractall(WORK, members=names)
        print(f"resumed {len(names)} result files from {z}")
!pip -q install uv
!uv venv -q --python 3.12 .venv
!uv pip install -q --python .venv/bin/python -e ".[uma,cuda]"
!.venv/bin/python -c "import jax; print('JAX devices:', jax.devices())"

## 3. Hugging Face token (only needed for UMA)

In [ ]:
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if token:
    os.environ["HF_TOKEN"] = token
ok = token and subprocess.run([".venv/bin/python", "-c",
    "from huggingface_hub import hf_hub_download as d; d('facebook/UMA','checkpoints/uma-s-1p2.pt')"]).returncode == 0
if not ok:
    MODELS = [m for m in MODELS if not m.startswith("uma")]
    print("UMA not accessible here (no HF_TOKEN secret, or license not yet approved): skipping it.")
print("Models:", MODELS)

## 4. Pilot: method check + timings

In [ ]:
!.venv/bin/chipff plan
if RUN_PILOT:
    !CUDA_VISIBLE_DEVICES=0 .venv/bin/chipff thermal -m mace-mpa-0 orb-v3 --materials Si Cu MgO GaN --max-prod-ps 20 --tag thermal_pilot
    !.venv/bin/python scripts/thermal_status.py --tag thermal_pilot

## 5. Full run
One model per GPU, in parallel; then that model's robustness reruns (seeds, cell size, time step). Results are saved after every batch. Re-running skips what is already done.

In [ ]:
import subprocess, sys
n_gpu = len([l for l in subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines() if l.startswith("GPU")]) or 1
print("GPUs:", n_gpu)
if RUN_FULL:
    os.makedirs("logs", exist_ok=True)
    queues = [MODELS[i::n_gpu] for i in range(n_gpu)]   # e.g. [[mace, uma], [orb]]
    procs = []
    for gpu, models in enumerate(queues):
        if not models:
            continue
        steps = []
        for m in models:
            steps.append(f".venv/bin/chipff thermal -m {m} > logs/thermal_{m}.log 2>&1")
            if RUN_ROBUSTNESS:
                steps.append(f".venv/bin/chipff robustness --what md -m {m} > logs/robustness_{m}.log 2>&1")
        cmd = " && ".join(steps)
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
        procs.append((gpu, models, subprocess.Popen(["bash", "-c", cmd], env=env, start_new_session=True)))
        print(f"GPU {gpu}: {models}")
    deadline = T0 + TIME_BUDGET_H * 3600
    while any(p.poll() is None for _, _, p in procs):
        if time.time() > deadline:
            print("Time budget reached: stopping (finished batches are saved)")
            import signal
            for _, _, p in procs:
                try:
                    os.killpg(p.pid, signal.SIGTERM)   # the whole process group, not just bash
                except ProcessLookupError:
                    pass
            time.sleep(30)
            break
        time.sleep(60)
    for gpu, models, p in procs:
        print(f"GPU {gpu} {models}: exit {p.poll()}")
    !.venv/bin/python scripts/thermal_status.py --tag thermal

## 6. Package the results

In [ ]:
!cd {WORK} && zip -q -r {OUT}/chipff_thermal_results.zip results/raw/*/thermal*.json logs 2>/dev/null; ls -la {OUT}/*.zip
!cd {WORK} && .venv/bin/python scripts/thermal_status.py --tag thermal | tail -80